# DATASCAPE 2026 — Data Understanding
**E-Commerce Package Quality Control · Pemeriksaan data sebelum keputusan training**

Notebook mandiri untuk Google Colab atau Jupyter lokal. Jalankan sel berurutan dengan **Runtime → Run all** setelah mengisi konfigurasi sumber data. CPU cukup; GPU tidak diperlukan.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhnzzzwp/DATASCAPE/blob/Yohanes-testing/datascape_2026_data_understanding.ipynb)

Tautan Colab aktif setelah notebook di-push ke branch `Yohanes-testing`; sesuaikan branch jika berbeda.

**Tujuan:** memeriksa atribut asli, komposisi kelas, kualitas gambar, duplikasi, inkonsistensi label, dan risiko kebocoran evaluasi. Hasil digunakan untuk mereview keputusan data training. Notebook ini tidak melatih model, mengubah label, menghapus gambar, atau membuat submission.

**Sumber konteks:** dokumen lokal `DATASCAPE.rtfd/TXT.rtf` yang diberikan pengguna, dibaca 1 Oktober 2026. Ringkasan di bawah berasal dari dokumen tersebut, bukan verifikasi ulang aturan di situs lomba. Jika panitia memperbarui aturan, gunakan versi resmi terbaru.


In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


## 1. Hubungan business understanding → analytical approach → data understanding

| Tahap | Konteks dan keputusan |
|---|---|
| Business understanding | Foto QC paket yang heterogen perlu mendukung otomatisasi pemeriksaan kondisi kemasan. Kerusakan yang terlewat dapat mengurangi kualitas layanan; alarm keliru menambah beban pemeriksaan. Biaya aktual kedua kesalahan belum tersedia. |
| Analytical approach | **Supervised multiclass image classification**: satu foto dipetakan ke satu label `Aman`, `Rusak`, atau `Lainnya`. Target evaluasi adalah Macro F1. |
| Data understanding (notebook ini) | Memastikan data dan evaluasi layak sebelum memilih preprocessing, pembagian validation, penanganan imbalance, dan arsitektur model. |
| Keputusan setelah review | Tentukan perlakuan terhadap file gagal baca, duplikat, konflik label, kualitas foto, dan kelompok yang tidak boleh terpisah antara train/validation. |

Ringkasan business understanding ini diturunkan dari overview lomba yang tersedia. Jika tim memiliki dokumen business understanding terpisah, cocokkan asumsi dan indikator suksesnya pada lembar keputusan di bagian akhir.

### Definisi label dari dokumen lomba
- **Aman:** paket/dus utuh, mulus, dan layak kirim.
- **Rusak:** paket/dus memiliki kerusakan fisik, misalnya penyok, sobek, basah, atau hancur.
- **Lainnya:** kemasan non-standar / out-of-distribution, misalnya food packaging, kantong plastik belanjaan, atau paper bag. **Bukan label untuk foto buram atau label tidak diketahui.**

Unit observasi adalah **file gambar**, belum terbukti satu file sama dengan satu paket unik. Persentase kelas dalam dataset tidak otomatis mewakili tingkat kerusakan pada pengiriman nyata.


## 2. Aturan, penilaian, dan batas analisis

Ringkasan dokumen dan klarifikasi Q&A panitia per **2 Oktober 2026**:

| Ketentuan | Isi |
|---|---|
| Periode pengerjaan | 1–10 Oktober 2026 |
| Data | Hanya data yang disediakan panitia; augmentasi dari data resmi diperbolehkan |
| Submission | Maksimal 3 per hari; memilih 2 submission untuk skor akhir |
| Leaderboard | Public 30%, private 70%, sesuai deskripsi dokumen |
| Bobot penilaian | Skor Kaggle 80%, notebook 10%, write-up 10% |
| Deliverables | `NamaTim_Notebook.ipynb` dan `NamaTim_Writeup.pdf` |
| Tim | Semua anggota satu tim Kaggle; nama sesuai pendaftaran |
| Prediksi | Prediksi/inference manual dilarang; model eksternal tidak boleh langsung digunakan untuk inferensi atau relabeling test set |
| Pretrained model | Diperbolehkan dan tidak dibatasi, tetapi harus di-fine-tune pada data train resmi |
| CLIP/VLM | Image encoder CLIP atau VLM lain boleh digunakan tanpa bagian teks jika di-fine-tune pada data train resmi |
| Self-training | Pseudo-labeling otomatis dan self-training pada training set diperbolehkan; manual labeling tetap dilarang |
| Bantuan AI | Coding, brainstorming, debugging, filtering, dan pengolahan training set diperbolehkan, dengan **AI Declaration** |

Klarifikasi Q&A ini mempersempit bunyi dokumen awal yang dapat terbaca seolah semua model AI/VLM dilarang. Model klasifikasi yang sudah dilatih atau di-fine-tune pada train resmi boleh melakukan inferensi test untuk menghasilkan submission. Yang dilarang adalah memakai model eksternal/off-the-shelf secara langsung untuk memberi label, deskripsi, embedding, atau prediksi atas test set.

Catatan terpusat: `reports/competition_rules_update_2026-10-02.md`. Sumber daring: [Live Q&A Data Mining](https://padlet.com/DataScape/live-qna-data-mining-s0240x081ojyxsbn85hf).

**Macro F1** memberi bobot sama untuk ketiga kelas:

$$F1_c=\frac{2TP_c}{2TP_c+FP_c+FN_c},\qquad
\mathrm{MacroF1}=\frac{F1_{Aman}+F1_{Rusak}+F1_{Lainnya}}{3}.$$

Accuracy saja tidak cukup ketika kelas timpang. Saat evaluasi model nantinya, sertakan precision, recall, dan F1 per kelas. Kasus pembagi nol harus ditangani eksplisit, misalnya `zero_division=0`.

**Batas notebook ini:** piksel dan contoh visual hanya dari **training**; test diperiksa sebatas nama file dan template submission. Batas tersebut menjaga notebook eksplorasi ini bebas dari test leakage dan bukan larangan memakai model final yang sah untuk membuat submission.

### AI Declaration
Notebook disusun dengan bantuan **OpenAI Codex** untuk perancangan kode, dokumentasi, debugging, dan pemeriksaan data secara programatik. Tidak menggunakan AI/LLM/SLM/VLM untuk menghasilkan label atau prediksi gambar kompetisi. Tim wajib memeriksa hasil dan menyesuaikan deklarasi ini dengan seluruh penggunaan AI yang benar-benar dilakukan sebelum pengumpulan.


## 3. Setup dan sumber data

1. Buka notebook di Colab. Gunakan runtime CPU.
2. Pilih `DATA_SOURCE` pada sel konfigurasi:
   - `auto`: gunakan folder dataset lokal jika ditemukan; jika tidak, Colab meminta unggah ZIP resmi.
   - `folder`: isi `DATA_PATH` ke folder yang berisi `Train_Set`, `Test_Set`, dan `sample_submission.csv`.
   - `drive`: mount Google Drive, lalu isi `DATA_PATH` ke folder dataset atau ZIP di Drive.
   - `zip`: isi `ZIP_PATH`, atau biarkan kosong agar Colab meminta unggahan ZIP.
3. Unduh ZIP resmi melalui akun Kaggle Anda setelah menerima aturan kompetisi. Notebook tidak memerlukan kredensial Kaggle.
4. Hasil rinci disimpan di `artifacts/data_understanding/`. Di Colab folder runtime bersifat sementara; unduh hasil yang diperlukan sebelum runtime dihapus.

Tidak perlu meng-upload dataset ke GitHub. ZIP hanya boleh berisi data kompetisi resmi.


In [ ]:
import importlib.util
import subprocess
import sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
            'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import os
import platform
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import f1_score
import importlib.metadata

SEED = 42
CLASSES = ['Aman', 'Rusak', 'Lainnya']
EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IN_COLAB = 'google.colab' in sys.modules
np.random.seed(SEED)
versions = {p: importlib.metadata.version(p) for p in required.values()}
print('Python:', platform.python_version(), '| Colab:', IN_COLAB)
display(pd.Series(versions, name='versi runtime'))


In [ ]:
# Ubah hanya konfigurasi ini untuk sumber data Anda.
DATA_SOURCE = 'auto'  # auto | folder | drive | zip
DATA_PATH = ''        # contoh: /content/drive/MyDrive/DATASCAPE/dataset
ZIP_PATH = ''         # contoh: /content/drive/MyDrive/DATASCAPE/dataset.zip
WORKERS = 4
OUTPUT_DIR = Path.cwd() / 'artifacts' / 'data_understanding'
EXTRACT_DIR = Path.cwd() / 'data' / 'datascape_extracted'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_SOURCE in {'auto', 'folder', 'drive', 'zip'}
assert 1 <= WORKERS <= 8


In [ ]:
def extract_official_zip(archive):
    # Validate members before extraction; refuse links/path traversal.
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    destination = EXTRACT_DIR.resolve()
    with zipfile.ZipFile(archive) as zf:
        for entry in zf.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('ZIP memuat path di luar folder ekstraksi.')
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP berisi symbolic link; gunakan arsip resmi biasa.')
            if target.exists() and not entry.is_dir():
                raise FileExistsError(f'Ekstraksi tidak menimpa file lama: {target}. Gunakan DATA_SOURCE=folder.')
        zf.extractall(destination)
    return destination


def upload_zip():
    if not IN_COLAB:
        raise FileNotFoundError('Isi DATA_PATH atau ZIP_PATH untuk eksekusi lokal.')
    from google.colab import files
    print('Unggah satu ZIP resmi dataset. Untuk data besar, gunakan opsi drive.')
    uploaded = files.upload()
    names = list(uploaded)
    del uploaded
    if len(names) != 1 or not names[0].lower().endswith('.zip'):
        raise ValueError('Diperlukan tepat satu file ZIP.')
    return Path(names[0])


if DATA_SOURCE == 'drive':
    if not IN_COLAB:
        raise RuntimeError('Opsi drive ditujukan untuk Google Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
    if not DATA_PATH:
        raise ValueError('Isi DATA_PATH ke folder atau ZIP di Google Drive.')
    base = Path(DATA_PATH).expanduser()
elif DATA_SOURCE == 'folder':
    if not DATA_PATH:
        raise ValueError('Isi DATA_PATH terlebih dahulu.')
    base = Path(DATA_PATH).expanduser()
elif DATA_SOURCE == 'zip':
    base = Path(ZIP_PATH).expanduser() if ZIP_PATH else upload_zip()
else:
    candidates = [Path(DATA_PATH).expanduser()] if DATA_PATH else [
        Path.cwd() / 'datascape-2026-data-mining-competition', EXTRACT_DIR]
    base = next((p for p in candidates if p.exists()), None)
    if base is None:
        base = Path(ZIP_PATH).expanduser() if ZIP_PATH else upload_zip()

if base.is_file() and base.suffix.lower() == '.zip':
    base = extract_official_zip(base)
if not base.is_dir():
    raise FileNotFoundError(f'Folder dataset tidak ditemukan: {base}')

# A wrapper directory inside the ZIP is supported; ambiguous roots are rejected.
roots = [base] if (base / 'Train_Set').is_dir() else [p.parent for p in base.rglob('Train_Set') if p.is_dir()]
roots = [p for p in roots if (p / 'Test_Set').is_dir() and (p / 'sample_submission.csv').is_file()]
if len(roots) != 1:
    raise ValueError(f'Diharapkan satu root dataset lengkap, ditemukan {len(roots)}. Isi DATA_PATH lebih spesifik.')
DATA_ROOT = roots[0].resolve()
print('Dataset:', DATA_ROOT)
print('Output lokal:', OUTPUT_DIR)


## 4. Inventaris dan atribut asli

**Pertanyaan:** apakah isi dataset sesuai struktur yang dijelaskan? Adakah metadata tambahan? Apakah setiap file training memiliki label folder yang valid?

Nama file/path adalah identitas teknis, bukan fitur semantik yang disarankan untuk model. Label training berasal dari folder. Kolom `label` pada sample submission adalah template keluaran, bukan ground truth test.


In [ ]:
all_files = sorted(p for p in DATA_ROOT.rglob('*') if p.is_file())
train_paths = sorted(p for p in (DATA_ROOT / 'Train_Set').rglob('*') if p.is_file() and p.suffix.lower() in EXT)
test_paths = sorted(p for p in (DATA_ROOT / 'Test_Set').rglob('*') if p.is_file() and p.suffix.lower() in EXT)
train_df = pd.DataFrame([{'filepath': str(p.relative_to(DATA_ROOT)), 'filename': p.name,
                         'label': p.relative_to(DATA_ROOT / 'Train_Set').parts[0]} for p in train_paths])
assert len(train_df), 'Tidak ada gambar training.'
assert set(train_df.label).issubset(CLASSES), 'Ada folder label training tak dikenal.'
assert set(train_df.label) == set(CLASSES), 'Ada kelas training yang tidak ditemukan.'
assert train_df.filepath.is_unique
submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv', dtype=str, keep_default_na=False)
assert list(submission.columns) == ['filename', 'label'], 'Periksa perubahan schema submission.'
metadata_files = [str(p.relative_to(DATA_ROOT)) for p in all_files if p.suffix.lower() in {'.csv','.json','.xlsx','.parquet','.txt'}]
test_names = [p.name for p in test_paths]
structure_checks = {
    'training_images': len(train_df), 'test_images': len(test_paths),
    'submission_rows': len(submission),
    'submission_duplicate_ids': int(submission.filename.duplicated().sum()),
    'test_duplicate_filenames': len(test_names) - len(set(test_names)),
    'missing_test_files': sorted(set(submission.filename)-set(test_names)),
    'unlisted_test_files': sorted(set(test_names)-set(submission.filename)),
    'empty_submission_ids': int(submission.filename.str.strip().eq('').sum())
}
print('Metadata yang tersedia:', metadata_files)
display(pd.Series(structure_checks, name='hasil struktur'))
display(train_df.head())
print('Nilai pada template (BUKAN label sebenarnya):')
display(submission.label.value_counts())
if (structure_checks['submission_duplicate_ids'] or structure_checks['test_duplicate_filenames']
    or structure_checks['missing_test_files'] or structure_checks['unlisted_test_files']
    or structure_checks['empty_submission_ids']):
    raise ValueError('Struktur test/template tidak konsisten. Periksa sebelum melanjutkan.')


### Kamus atribut

| Atribut | Sumber / definisi | Batas interpretasi |
|---|---|---|
| `filepath`, `filename` | File training lokal | Bukan identitas paket unik |
| `label` | Folder kelas | Dapat mengandung inkonsistensi |
| `file_bytes`, `format`, `mode` | Ukuran byte, format aktual, mode warna | Ekstensi belum tentu sesuai format |
| `width`, `height`, `aspect_ratio` | Ukuran setelah orientasi EXIF, W/H | Resolusi foto, bukan dimensi fisik paket |
| `brightness`, `contrast` | Rata-rata dan simpangan baku grayscale /255 | Dipengaruhi objek dan latar |
| `dark_pixel_fraction`, `bright_pixel_fraction` | Proporsi grayscale ≤0,05 dan ≥0,95 | Kandidat masalah pencahayaan, bukan diagnosis |
| `sharpness_proxy` | Varians Laplacian pada grayscale resize 128×128 | Dipengaruhi tekstur/resizing; bukan label blur tervalidasi |
| `file_sha256`, `pixel_sha256` | Hash byte dan piksel RGB beserta ukuran | Mendeteksi duplikat persis, bukan near-duplicate |
| `exif_present` | Keberadaan metadata EXIF | Tidak membaca/menganggap ada lokasi atau waktu pengiriman |
| `error` | Pesan gagal baca/decode | Dipisahkan untuk review, file asli tetap ada |

Gambar RGBA/palet dikonversi ke RGB untuk statistik dan hashing; transparansi diabaikan, bukan dikomposisikan ke latar tertentu. Definisi hash ini harus dibaca sebagai kesamaan representasi RGB yang digunakan audit.


## 5. Distribusi kelas dan implikasi metrik

**Pertanyaan:** seberapa timpang kelas? Apakah accuracy tinggi masih bisa menyembunyikan performa buruk pada kelas minoritas?

Prediksi kelas mayoritas berikut hanyalah demonstrasi numerik pada label training, bukan model, bukan evaluasi generalisasi, dan tidak digunakan untuk test.


In [ ]:
class_counts = train_df.label.value_counts().reindex(CLASSES, fill_value=0)
class_table = pd.DataFrame({'jumlah': class_counts, 'persen': class_counts / len(train_df) * 100})
display(class_table.round(2))
majority = class_counts.idxmax()
majority_predictions = [majority] * len(train_df)
print('Accuracy dummy mayoritas:', round(class_counts.max()/len(train_df), 4))
print('Macro F1 dummy mayoritas:', round(f1_score(train_df.label, majority_predictions,
                                                labels=CLASSES, average='macro', zero_division=0), 4))
ax = class_counts.plot.bar(color=['#27806a','#c05a48','#687caf'], rot=0, figsize=(7,4))
ax.set(title='Komposisi file training', xlabel='Label folder', ylabel='Jumlah gambar')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=150); plt.show()


## 6. Audit seluruh gambar training

Audit ini membaca semua piksel training dan menghitung hash; perlu beberapa menit bergantung CPU dan lokasi dataset. Memuat langsung dari Drive biasanya lebih lambat daripada ZIP yang diekstrak ke runtime Colab.

Tidak ada gambar test yang dibuka. File gagal baca dicatat tanpa menghapus file asli. Semua statistik berikut menggunakan gambar training yang berhasil dibaca.


In [ ]:
def inspect(path):
    row = {'filepath': str(path.relative_to(DATA_ROOT)), 'filename': path.name,
           'label': path.relative_to(DATA_ROOT / 'Train_Set').parts[0]}
    try:
        raw = path.read_bytes()
        row.update(file_bytes=len(raw), file_sha256=hashlib.sha256(raw).hexdigest())
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            row.update(format=im.format, mode=im.mode, exif_present=bool(im.getexif()))
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            w, h = rgb.size
            pixel_hash = hashlib.sha256(f'{w}x{h}:RGB:'.encode() + rgb.tobytes()).hexdigest()
            gray = np.asarray(rgb.convert('L'), dtype=np.float32) / 255
            small = np.asarray(rgb.resize((128, 128)).convert('L'), dtype=np.float32) / 255
            lap = (small[:-2, 1:-1] + small[2:, 1:-1] + small[1:-1, :-2]
                   + small[1:-1, 2:] - 4 * small[1:-1, 1:-1])
            row.update(width=w, height=h, aspect_ratio=w/h,
                       brightness=float(gray.mean()), contrast=float(gray.std()),
                       dark_pixel_fraction=float((gray <= 0.05).mean()),
                       bright_pixel_fraction=float((gray >= 0.95).mean()),
                       sharpness_proxy=float(lap.var()), pixel_sha256=pixel_hash,
                       error='')
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    return row


In [ ]:
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    records = list(tqdm(pool.map(inspect, train_paths), total=len(train_paths), desc='Audit training'))
audit_df = pd.DataFrame(records)
good = audit_df[audit_df.error.eq('')].copy().reset_index(drop=True)
bad = audit_df[audit_df.error.ne('')].copy()
assert len(good) + len(bad) == len(train_df)
assert len(good), 'Tidak ada gambar training yang dapat dibaca.'
print(f'Total: {len(audit_df):,}; terbaca: {len(good):,}; gagal baca: {len(bad):,}')
display(bad[['filepath','label','error']])
audit_df.to_csv(OUTPUT_DIR / 'training_attributes.csv', index=False)
bad.to_csv(OUTPUT_DIR / 'unreadable_images.csv', index=False)


## 7. Kualitas gambar dan perbedaan antarkelas

Threshold berikut untuk **menyaring kandidat review**, bukan menentukan foto rusak, label paket, atau aturan pembuangan otomatis. Foto terang dapat berisi latar putih; gambar minim tekstur dapat memiliki proksi ketajaman rendah.


In [ ]:
metrics = ['width','height','aspect_ratio','brightness','contrast','sharpness_proxy']
display(good[metrics].describe().T.round(4))
display(good.groupby('label')[metrics].median().reindex(CLASSES).round(4))
print('Format aktual per kelas:'); display(pd.crosstab(good.label, good.format).reindex(CLASSES))
print('Mode warna:'); display(good['mode'].value_counts())
print('Gambar dengan EXIF:', int(good.exif_present.sum()))

DARK_MEAN = 0.2
BRIGHT_MEAN = 0.8
MIN_SHORT_SIDE = 224
ASPECT_LIMITS = (0.5, 2.0)
quality = good[['filepath','filename','label']].copy()
quality['dark_candidate'] = good.brightness < DARK_MEAN
quality['bright_candidate'] = good.brightness > BRIGHT_MEAN
quality['low_resolution'] = good[['width','height']].min(axis=1) < MIN_SHORT_SIDE
quality['extreme_aspect'] = ~good.aspect_ratio.between(*ASPECT_LIMITS)
flags = ['dark_candidate','bright_candidate','low_resolution','extreme_aspect']
print('Jumlah kandidat (dapat tumpang tindih):'); display(quality[flags].sum())
print('Persentase kandidat dalam tiap kelas:')
display((quality.groupby('label')[flags].mean().reindex(CLASSES)*100).round(2))
quality.to_csv(OUTPUT_DIR / 'quality_review_flags.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(12,8))
for ax, metric in zip(axes.flat, ['width','aspect_ratio','brightness','sharpness_proxy']):
    ax.boxplot([good.loc[good.label.eq(c), metric] for c in CLASSES], showfliers=False)
    ax.set_xticks(range(1, len(CLASSES)+1), CLASSES)
    ax.set_title(metric + ' menurut kelas')
    if metric in {'width','sharpness_proxy'}:
        ax.set_yscale('log')
fig.suptitle('Outlier tidak ditampilkan pada boxplot; tetap disimpan dalam tabel audit')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'quality_by_class.png', dpi=150); plt.show()


### Cara membaca perbedaan

Perbedaan resolusi, format, atau pencahayaan antarkelas bisa mengindikasikan perbedaan proses pengumpulan data. Ini **belum membuktikan model memanfaatkan shortcut**, karena model belum dilatih. Gunakan temuan untuk merancang evaluasi per kelompok kualitas dan memeriksa konsistensi preprocessing.

Atribut kurir, rute, waktu pengiriman, biaya kerusakan, dan material kemasan tidak boleh disimpulkan dari statistik piksel. Analisis risiko operasional tersebut memerlukan metadata tambahan.


## 8. Contoh visual training untuk memahami variasi data

Sampel dipilih deterministik per label folder. Judul mempertahankan label yang diberikan dataset; tampilan ini tidak membuat label baru. Periksa keterlihatan kemasan, latar, pencahayaan, serta variasi definisi kelas. Simpan catatan pengamatan di lembar keputusan; jangan mengubah label tanpa keputusan tim dan pengecekan aturan.


In [ ]:
def show_training_rows(rows, title, columns=4):
    rows = list(rows)
    if not rows:
        print('Tidak ada contoh untuk', title)
        return
    nrows = int(np.ceil(len(rows)/columns))
    fig, axes = plt.subplots(nrows, columns, figsize=(3.4*columns, 3.3*nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, row in zip(axes.flat, rows):
        path = (DATA_ROOT / row['filepath']).resolve()
        assert path.is_relative_to((DATA_ROOT / 'Train_Set').resolve()), 'Hanya training boleh ditampilkan.'
        with Image.open(path) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(f"{row['label']} | {row['filename']}", fontsize=8)
    fig.suptitle(title); fig.tight_layout(); plt.show()

sample_rows = []
for label in CLASSES:
    subset = good[good.label.eq(label)]
    sample_rows.extend(subset.sample(min(4, len(subset)), random_state=SEED).to_dict('records'))
show_training_rows(sample_rows, 'Contoh training — label asli dari folder')


## 9. Duplikasi persis dan inkonsistensi label

Hash file mendeteksi byte identik. Hash piksel mencakup dimensi dan RGB setelah orientasi EXIF sehingga dapat menangkap isi RGB identik walau metadata file berbeda. **Near-duplicate** (crop, kompresi, perubahan sudut) belum tercakup.

Kelompok dengan lebih dari satu label adalah bukti inkonsistensi anotasi. Pemeriksaan ini tidak menentukan label mana yang benar.


In [ ]:
def duplicate_summary(frame, key):
    groups = frame.groupby(key).agg(images=('filepath','size'), label_count=('label','nunique'))
    duplicate_groups = groups[groups.images.gt(1)]
    return {'groups': len(duplicate_groups), 'images_in_groups': int(duplicate_groups.images.sum()),
            'redundant_copies': int((duplicate_groups.images-1).sum()),
            'conflicting_groups': int(duplicate_groups.label_count.gt(1).sum())}

dup_summary = {key: duplicate_summary(good, key) for key in ['file_sha256','pixel_sha256']}
display(pd.DataFrame(dup_summary).T)
pixel_groups = good.groupby('pixel_sha256').agg(images=('filepath','size'), labels=('label','nunique'))
conflict_hashes = set(pixel_groups.index[pixel_groups.labels.gt(1)])
conflicts = good[good.pixel_sha256.isin(conflict_hashes)].sort_values(['pixel_sha256','label','filename'])
duplicates = good[good.duplicated('pixel_sha256', keep=False)].sort_values(['pixel_sha256','filename'])
print('Contoh bukti konflik label:')
display(conflicts[['pixel_sha256','filename','label']].head(16))
duplicates.to_csv(OUTPUT_DIR / 'exact_duplicates.csv', index=False)
conflicts.to_csv(OUTPUT_DIR / 'label_conflicts.csv', index=False)
print('Jumlah gambar dalam kelompok konflik:', len(conflicts))


## 10. Diagnosis split: path berbeda belum tentu isi berbeda

Simulasi pertama memakai stratified random split 80/20 pada file yang terbaca, seed 42. **Urutan input di notebook ini disortir menurut path**, sehingga angka overlap dapat berbeda dari baseline lama yang memakai urutan filesystem. Tidak ada training model.

Simulasi kedua adalah **kandidat**, bukan penetapan data training: kelompok konflik disisihkan sementara dari simulasi; duplikat yang tersisa diberi group berdasarkan hash piksel. Fold pertama dari StratifiedGroupKFold lima fold memberi validation sekitar 20%, menjaga kelompok identik tidak menyeberang split. Distribusi dan ukuran fold tidak selalu tepat 80/20.

Tidak ada data asli dihapus atau label diubah. Near-duplicate dan foto berbeda dari paket yang sama masih dapat bocor; hash persis saja belum menyelesaikan seluruh risiko.


In [ ]:
def split_diagnostics(train_part, val_part):
    common = set(train_part.pixel_sha256) & set(val_part.pixel_sha256)
    return {'train_images': len(train_part), 'validation_images': len(val_part),
            'shared_hash_groups': len(common),
            'overlapping_validation_images': int(val_part.pixel_sha256.isin(common).sum()),
            'overlapping_validation_percent': float(val_part.pixel_sha256.isin(common).mean()*100)}

split_results = {}
random_train, random_val = pd.DataFrame(), pd.DataFrame()
if good.label.nunique() == len(CLASSES) and good.label.value_counts().min() >= 2:
    random_train, random_val = train_test_split(good, test_size=.2, stratify=good.label, random_state=SEED)
    split_results['random_stratified'] = split_diagnostics(random_train, random_val)
else:
    print('Data per kelas terlalu sedikit untuk simulasi random split.')

candidate = good[~good.pixel_sha256.isin(conflict_hashes)].copy().reset_index(drop=True)
group_counts = candidate.groupby('label').pixel_sha256.nunique().reindex(CLASSES, fill_value=0)
print('Kelompok unik per kelas setelah konflik disisihkan dari simulasi:')
display(group_counts)
if group_counts.min() >= 5:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    ti, vi = next(splitter.split(candidate, candidate.label, groups=candidate.pixel_sha256))
    grouped_train, grouped_val = candidate.iloc[ti], candidate.iloc[vi]
    split_results['grouped_candidate'] = split_diagnostics(grouped_train, grouped_val)
    assert split_results['grouped_candidate']['shared_hash_groups'] == 0
    display(pd.DataFrame({'train': grouped_train.label.value_counts(),
                          'validation': grouped_val.label.value_counts()}).reindex(CLASSES).fillna(0))
else:
    print('Minimal 5 kelompok per kelas diperlukan untuk kandidat lima fold.')
display(pd.DataFrame(split_results).T)
print('Simulasi ini belum mengunci split atau memilih perlakuan duplikat untuk training.')


## 11. Ringkasan otomatis dan kandidat tindak lanjut

Ringkasan dihitung dari eksekusi saat ini, bukan angka yang diketik dari audit sebelumnya. Peringatan menunjukkan pekerjaan review, bukan instruksi otomatis mengubah dataset.


In [ ]:
findings = []
findings.append(f'Dataset memuat {len(train_df):,} training dan {len(test_paths):,} test; test hanya diperiksa nama file.')
findings.append(f'Kelas terbanyak: {class_counts.idxmax()} ({class_counts.max()/len(train_df):.2%}); paling sedikit: {class_counts.idxmin()} ({class_counts.min()/len(train_df):.2%}).')
findings.append(f'{len(bad):,} gambar gagal dibaca; {len(good):,} masuk statistik piksel.')
pd_stats = dup_summary['pixel_sha256']
findings.append(f"Duplikat RGB persis: {pd_stats['groups']:,} kelompok, {pd_stats['images_in_groups']:,} gambar, {pd_stats['conflicting_groups']:,} kelompok lintas label.")
if 'random_stratified' in split_results:
    result = split_results['random_stratified']
    findings.append(f"Simulasi split acak: {result['overlapping_validation_images']:,} gambar validation ({result['overlapping_validation_percent']:.2f}%) identik dengan training.")
for sentence in findings:
    display(Markdown('- ' + sentence))

review_actions = pd.DataFrame([
    ['Kualitas file', f'{len(bad)} gagal baca', 'Periksa sumber resmi; putuskan karantina/perbaikan'],
    ['Label', f'{len(conflict_hashes)} kelompok konflik', 'Tinjau training dan aturan; jangan memilih label otomatis'],
    ['Duplikat', f"{pd_stats['redundant_copies']} salinan berlebih", 'Putuskan deduplikasi atau grouping; pertimbangkan bobot sampel'],
    ['Kualitas foto', f"{int(quality.low_resolution.sum())} sisi pendek <{MIN_SHORT_SIDE}px", 'Bandingkan resize/padding; jangan membuang berdasarkan ambang saja'],
    ['Bias sumber', 'Lihat distribusi format, resolusi, dan brightness', 'Rencanakan evaluasi model per kelompok kualitas'],
    ['Near-duplicate', 'Belum diuji', 'Pertimbangkan perceptual hash + review kandidat sebelum split final'],
], columns=['area','bukti saat ini','keputusan yang perlu direview'])
display(review_actions)
review_actions.to_csv(OUTPUT_DIR / 'review_actions.csv', index=False)


## 12. Insight selain klasifikasi: tersedia vs belum teruji

| Analisis | Bisa digunakan sekarang | Batas / kebutuhan berikutnya |
|---|---|---|
| Audit konsistensi label | Tabel konflik berbasis hash | Tidak menentukan label benar |
| Deduplikasi dan leakage | Kelompok gambar identik, simulasi split | Near-duplicate dan identitas paket perlu pemeriksaan tambahan |
| Kualitas akuisisi gambar | Resolusi, brightness, contrast, proksi sharpness | Ambang perlu divalidasi sebelum menjadi standar foto QC |
| Bias distribusi | Perbandingan antarkelas | Belum membuktikan shortcut model atau sumber penyebab |
| Clustering / similarity search | Dapat dirancang setelah audit | Belum dijalankan; hasil cluster bukan label baru |
| Anomaly detection | Dapat dirancang dari fitur training | Anomali bukan otomatis paket rusak |
| Lokalisasi / tingkat keparahan | Belum tersedia anotasi target | Perlu anotasi dan evaluasi tambahan sesuai izin panitia |
| Risiko per kurir / rute | Belum ada metadata yang mendukung | Perlu data operasional; tidak boleh menambah data eksternal untuk kompetisi |

Insight tambahan membantu memahami data; tujuan kompetisi tetap klasifikasi tiga kelas.


## 13. Lembar review keputusan sebelum training

Isi bagian ini **setelah meninjau tabel dan contoh**, lalu cocokkan dengan business understanding tim. Status awal sengaja belum diputuskan.

| Keputusan | Bukti yang ditinjau | Pilihan / alasan tim | Status |
|---|---|---|---|
| File gagal baca | `unreadable_images.csv` | … | Belum diputuskan |
| Kelompok label konflik | `label_conflicts.csv` | … | Belum diputuskan |
| Duplikat satu label | `exact_duplicates.csv` | Pertahankan berkelompok / satu representatif / alternatif beserta alasan | Belum diputuskan |
| Near-duplicate | Belum diuji | Metode kandidat dan cara review | Belum diputuskan |
| Validation split | Simulasi acak vs grouped | Unit grouping, seed, proporsi, kelas tiap split | Belum diputuskan |
| Resolusi input | Distribusi ukuran dan contoh training | Ukuran, resize langsung / padding, dampak pada detail cacat | Belum diputuskan |
| Foto terang/gelap/buram | Statistik dan review training | Pertahankan / augmentasi / filter tervalidasi | Belum diputuskan |
| Imbalance | Distribusi dan Macro F1 dummy | Class weight / sampler / tanpa koreksi sebagai baseline | Belum diputuskan |
| Kriteria sukses | Macro F1 dan kebutuhan bisnis | Target eksperimen, precision/recall per kelas | Belum diputuskan |
| Kepatuhan & AI Declaration | Dokumen resmi dan deklarasi aktual | Nama reviewer, tanggal, penyesuaian | Belum diputuskan |

**Sebelum training:** dokumentasikan keputusan; kunci validation split setelah grouping; gunakan training untuk fitting transform/statistik/fitur model; jangan memilih label atau model memakai isi test. Jika validation digunakan berulang untuk tuning, catat risiko overfitting terhadap validation.

**Catatan reviewer:**
- Nama tim / reviewer: …
- Tanggal review: …
- Temuan yang belum cocok dengan ekspektasi: …
- Keputusan untuk eksperimen pertama: …


## 14. Ekspor bukti lokal dan persiapan commit GitHub

Folder output berisi atribut per gambar, temuan konflik/duplikasi, flag kualitas, ringkasan JSON, versi library, dan grafik agregat. Tidak ada ekspor split final atau submission.

Untuk GitHub, commit **notebook dengan output dibersihkan** dan `.gitignore`. Setelah menjalankan notebook di Colab, gunakan **Edit → Clear all outputs** sebelum menyimpan salinan yang akan di-commit; ini mencegah contoh gambar training tersimpan di dalam `.ipynb`. Periksa juga diff sebelum commit. Untuk pengumpulan lomba, ikuti ketentuan panitia mengenai output notebook.

Aturan `.gitignore` yang diperlukan pada repo:
```gitignore
/datascape-2026-data-mining-competition/
/datascape-2026-data-mining-competition.zip
/data/
/datasets/
/artifacts/
.ipynb_checkpoints/
```
Jika menyimpan dataset/ZIP dengan nama atau lokasi lain, tambahkan lokasi itu ke `.gitignore`. Gitignore tidak menghapus file yang sudah terlacak; cek `git ls-files` sebelum commit. Jangan memakai `git add -f` untuk dataset.


In [ ]:
summary = {
    'scope': 'Training pixels only; test filenames/template only; no training or predictions.',
    'structure': structure_checks,
    'class_counts': {str(k): int(v) for k,v in class_counts.items()},
    'readable': len(good), 'unreadable': len(bad),
    'duplicates': dup_summary, 'split_simulations': split_results,
    'quality_thresholds': {'dark_mean': DARK_MEAN, 'bright_mean': BRIGHT_MEAN,
                           'min_short_side': MIN_SHORT_SIDE, 'aspect_limits': ASPECT_LIMITS},
    'quality_flag_counts': {k: int(quality[k].sum()) for k in flags},
    'class_medians': good.groupby('label')[metrics].median().to_dict(orient='index'),
    'findings': findings, 'versions': versions, 'seed': SEED,
    'review_status': 'Pending human review; no final training-data decisions applied.'
}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Hasil audit tersimpan:', OUTPUT_DIR)
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))
print('Selesai. Review bukti dan isi lembar keputusan sebelum melanjutkan training.')


## 15. Audit mendalam metadata, warna, embedding, dan near-duplicate

Ekstensi audit yang diminta tersedia pada `scripts/deep_image_audit.py`. Jalankan setelah tabel `artifacts/data_understanding/training_attributes.csv` terbentuk. Audit menambahkan mode/kanal warna, orientasi EXIF, dimensi, luminance, kontras, dynamic range, saturasi, color cast, perceptual hash empat rotasi, embedding visual, 15-NN label review, dan t-SNE.

```bash
python scripts/deep_image_audit.py
```

Hasil lengkap dan batas interpretasi dijelaskan pada `reports/deep_data_understanding.md`. Semua flag adalah kandidat review; file training dan label asli tidak diubah.

**Batas kepatuhan:** embedding pralatih pada audit ini hanya eksplorasi. Untuk relabeling kompetisi, bangun ulang sinyal dengan prediksi out-of-fold model yang dilatih dari nol hanya pada training resmi.


In [ ]:
deep_dir = Path.cwd() / 'artifacts' / 'deep_data_understanding'
if (deep_dir / 'summary.json').exists():
    deep_summary = json.loads((deep_dir / 'summary.json').read_text(encoding='utf-8'))
    display(pd.Series(deep_summary, name='deep audit'))
    display(Markdown((deep_dir / 'REPORT.md').read_text(encoding='utf-8')))
else:
    print('Jalankan: python scripts/deep_image_audit.py')
